In [1]:
!nvidia-smi

Thu Apr 30 00:00:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             55W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memory GB:", torch.cuda.get_device_properties(0).total_memory / 1e9)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
CUDA build: 12.8
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Memory GB: 85.094825984


In [3]:
from pathlib import Path
import os

repo_dir = Path("/content/modded-nanogpt")
repo_url = "https://github.com/bluepeach1121/modded-nanogpt.git"
branch = "modded-gpt-test"

if repo_dir.exists():
    print("Repo already exists. Updating existing clone.")
    os.chdir(repo_dir)
    !git fetch origin
    !git checkout {branch}
    !git pull
else:
    print("Repo not found. Cloning.")
    %cd /content
    !git clone {repo_url}
    %cd /content/modded-nanogpt
    !git checkout {branch}

print("\nCurrent branch:")
!git branch

print("\nStatus:")
!git status

Repo not found. Cloning.
/content
Cloning into 'modded-nanogpt'...
remote: Enumerating objects: 7852, done.
remote: Counting objects: 100% (1020/1020), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 7852 (delta 935), reused 866 (delta 865), pack-reused 6832 (from 4)
Receiving objects: 100% (7852/7852), 38.47 MiB | 33.64 MiB/s, done.
Resolving deltas: 100% (5181/5181), done.
/content/modded-nanogpt
Branch 'modded-gpt-test' set up to track remote branch 'modded-gpt-test' from 'origin'.
Switched to a new branch 'modded-gpt-test'

Current branch:
  master
* modded-gpt-test

Status:
On branch modded-gpt-test
Your branch is up to date with 'origin/modded-gpt-test'.

nothing to commit, working tree clean


In [4]:
from pathlib import Path

%cd /content/modded-nanogpt

paths = [
    "data/cached_fineweb10B.py",
    "records/track_3_optimization/train_gpt_simple.py",
    "records/track_3_optimization/README.md",
]

for p in paths:
    path = Path(p)
    print(f"{p}: {path.exists()}")

/content/modded-nanogpt
data/cached_fineweb10B.py: True
records/track_3_optimization/train_gpt_simple.py: True
records/track_3_optimization/README.md: True


In [5]:
%cd /content/modded-nanogpt
!python data/cached_fineweb10B.py 40

/content/modded-nanogpt
fineweb_val_000000.bin: 100% 200M/200M [00:02<00:00, 98.6MB/s]
fineweb_train_000001.bin: 100% 200M/200M [00:02<00:00, 90.5MB/s]
fineweb_train_000002.bin: 100% 200M/200M [00:02<00:00, 76.6MB/s] 
fineweb_train_000003.bin: 100% 200M/200M [00:01<00:00, 142MB/s] 
fineweb_train_000004.bin: 100% 200M/200M [00:02<00:00, 82.9MB/s] 
fineweb_train_000005.bin: 100% 200M/200M [00:01<00:00, 124MB/s]
fineweb_train_000006.bin: 100% 200M/200M [00:01<00:00, 142MB/s] 
fineweb_train_000007.bin: 100% 200M/200M [00:02<00:00, 99.5MB/s]
fineweb_train_000008.bin: 100% 200M/200M [00:01<00:00, 124MB/s] 
fineweb_train_000009.bin: 100% 200M/200M [00:02<00:00, 76.6MB/s] 
fineweb_train_000010.bin: 100% 200M/200M [00:02<00:00, 90.5MB/s]
fineweb_train_000011.bin: 100% 200M/200M [00:01<00:00, 142MB/s]
fineweb_train_000012.bin: 100% 200M/200M [00:01<00:00, 142MB/s] 
fineweb_train_000013.bin: 100% 200M/200M [00:02<00:00, 99.5MB/s]
fineweb_train_000014.bin: 100% 200M/200M [00:02<00:00, 82.9MB/s]
fi

In [6]:
from pathlib import Path

data_dir = Path("/content/modded-nanogpt/data/fineweb10B")

print("Data dir exists:", data_dir.exists())
print("Data dir:", data_dir)

if data_dir.exists():
    train_files = sorted(data_dir.glob("fineweb_train_*.bin"))
    val_files = sorted(data_dir.glob("fineweb_val_*.bin"))
    all_bin_files = sorted(data_dir.glob("*.bin"))

    total_gb = sum(f.stat().st_size for f in all_bin_files) / 1e9

    print("\nTrain shards:", len(train_files))
    print("Val shards:", len(val_files))
    print("Total .bin files:", len(all_bin_files))
    print(f"Total size: {total_gb:.3f} GB")

    if train_files:
        print("\nFirst train shard:", train_files[0].name)
        print("Last train shard:", train_files[-1].name)

    if val_files:
        print("Validation shard:", val_files[0].name)

    print("\nLast 10 train shards:")
    for f in train_files[-10:]:
        print(f"{f.name}: {f.stat().st_size / 1e9:.3f} GB")

Data dir exists: True
Data dir: /content/modded-nanogpt/data/fineweb10B

Train shards: 40
Val shards: 1
Total .bin files: 41
Total size: 8.200 GB

First train shard: fineweb_train_000001.bin
Last train shard: fineweb_train_000040.bin
Validation shard: fineweb_val_000000.bin

Last 10 train shards:
fineweb_train_000031.bin: 0.200 GB
fineweb_train_000032.bin: 0.200 GB
fineweb_train_000033.bin: 0.200 GB
fineweb_train_000034.bin: 0.200 GB
fineweb_train_000035.bin: 0.200 GB
fineweb_train_000036.bin: 0.200 GB
fineweb_train_000037.bin: 0.200 GB
fineweb_train_000038.bin: 0.200 GB
fineweb_train_000039.bin: 0.200 GB
fineweb_train_000040.bin: 0.200 GB


In [7]:
from pathlib import Path
import re

script_path = Path("/content/modded-nanogpt/records/track_3_optimization/train_gpt_simple.py")
text = script_path.read_text()

match = re.search(r"train_steps\s*=\s*(\d+)", text)
print("Script exists:", script_path.exists())
print("train_steps:", match.group(1) if match else "not found")

print("\nMuon settings lines:")
for line in text.splitlines():
    if "optimizer2 = Muon" in line or "lr=0.025" in line or "weight_decay=0.0125" in line:
        print(line)

Script exists: True
train_steps: 3500

Muon settings lines:
optimizer2 = Muon([p for p in model.blocks.parameters() if p.ndim >= 2],
                  lr=0.025, weight_decay=0.0125)


In [8]:
%cd /content/modded-nanogpt
!torchrun --standalone --nproc_per_node=1 records/track_3_optimization/train_gpt_simple.py

/content/modded-nanogpt
logs/f31184c6-f0c1-414e-87c3-24665995cbc3.txt
step:0/3500 val_loss:10.82583 train_time:0.000s step_avg:0.07ms
step:1/3500 train_time:62.957s step_avg:62957.26ms
step:2/3500 train_time:65.493s step_avg:32746.49ms
step:3/3500 train_time:67.965s step_avg:22655.15ms
step:4/3500 train_time:70.431s step_avg:17607.79ms
step:5/3500 train_time:72.896s step_avg:14579.24ms
step:6/3500 train_time:75.361s step_avg:12560.14ms
step:7/3500 train_time:77.831s step_avg:11118.76ms
step:8/3500 train_time:80.299s step_avg:10037.35ms
step:9/3500 train_time:82.765s step_avg:9196.09ms
step:10/3500 train_time:85.230s step_avg:8523.00ms
step:11/3500 train_time:87.697s step_avg:7972.41ms
step:12/3500 train_time:90.166s step_avg:7513.87ms
step:13/3500 train_time:92.637s step_avg:7125.89ms
step:14/3500 train_time:95.103s step_avg:6793.09ms
step:15/3500 train_time:97.570s step_avg:6504.67ms
step:16/3500 train_time:100.036s step_avg:6252.26ms
step:17/3500 train_time:102.502s step_avg:6029.55m